In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# ---------------------------------------------------------
# 1. The Dataset Blueprint (Unchanged)
# ---------------------------------------------------------
class TrafficSignDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_path = self.data_frame.iloc[idx]['Path']
        full_path = os.path.abspath(img_path)
        image = Image.open(full_path).convert('RGB')
        
        y_label = torch.tensor(int(self.data_frame.iloc[idx]['ClassId']))
        
        if self.transform:
            image = self.transform(image)
            
        return (image, y_label)

# ---------------------------------------------------------
# 2. The Main Training Execution
# ---------------------------------------------------------
if __name__ == "__main__":
    # A. Setup Device
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # B. Setup DataLoaders (CRITICAL: Notice the 224x224 resize for ResNet!)
    # We also add Normalization, which is the standard color-scaling ResNet was trained on
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("Loading datasets...")
    train_dataset = TrafficSignDataset(csv_file='Train.csv', transform=transform)
    # Keeping num_workers=0 to prevent the macOS spawn error!
    train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True, num_workers=0)

    # C. Initialize the Pre-Trained Model (The Master Art Critic)
    print("Downloading and building ResNet18...")
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    # D. Freeze the Backbone (Don't ruin what it already knows)
    for param in model.parameters():
        param.requires_grad = False

    # E. Swap the Head (Give it a new 43-class brain)
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 43)
    
    # Move the upgraded model to your Mac's GPU
    model = model.to(device)
    
    # F. Setup Loss and Optimizer
    criterion = nn.CrossEntropyLoss()
    # Notice we ONLY hand the optimizer the new 'model.fc.parameters()', making it super fast
    optimizer = optim.Adam(model.fc.parameters(), lr=0.0001)

    # G. The Training Loop
    # Because it is pre-trained, it learns incredibly fast. We only need 5 epochs!
    epochs = 25
    print("\nStarting Transfer Learning Training... (This may take a few minutes)")
    
    for epoch in range(epochs):
        model.train() 
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward Pass
            outputs = model(images)
            loss = criterion(outputs, labels) 
            
            # Backward Pass
            optimizer.zero_grad() 
            loss.backward()       
            optimizer.step()
            
            # Tracking Progress
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        # Print stats at the end of each epoch
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

    print("\nTraining Complete!")
    
    # H. Save the new, highly accurate brain!
    torch.save(model.state_dict(), 'resnet_traffic_sign.pth')
    print("Model saved to 'resnet_traffic_sign.pth'")

Using device: mps
Loading datasets...

Starting Transfer Learning Training... (This may take a few minutes)
Epoch [1/25] | Loss: 2.7900 | Accuracy: 30.24%
Epoch [2/25] | Loss: 1.9040 | Accuracy: 53.80%
Epoch [3/25] | Loss: 1.5181 | Accuracy: 62.94%
